# Pre-S4 - Calidad de datos y particionamiento: un segundo caso de uso (sensores ambientales)

**Actividad:** construir el notebook `pre_s04_calidad_campo_electrico_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), aplicando las mismas tecnicas de calidad de datos de S3 (esquema, nulos, duplicados, particionado) sobre un dataset real distinto: tres fuentes de sensores ambientales que hay que integrar antes de poder tratarlas.

A diferencia de S3 (H&M, una sola fuente ya integrada), aca la integracion de multiples fuentes es en si misma parte del problema de calidad -- el mismo patron que se repite en cualquier proyecto real con mas de un sistema de origen.


## 1. Crear la `SparkSession`


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3b-calidad-campo-electrico")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 00:57:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/pre-s04-calidad-campo-electrico/data"
ARTIFACTS = "/opt/pre-s04-calidad-campo-electrico/artifacts"


## 2. Cargar las tres fuentes con esquema explicito

Tres archivos, cada uno con su propia estructura, extraidos previamente de tres sensores reales: campo electrico, campo magnetico, variables ambientales. Los tres comparten `FechaHora` como clave, medida minuto a minuto.


In [3]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType

schema_ce = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CE", DoubleType(), nullable=True),
])
df_ce = spark.read.csv(f"{ORIGEN_DATOS}/campo_electrico.csv", header=True, schema=schema_ce)
print(f"Campo electrico: {df_ce.count():,} registros")

schema_cm = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CM", DoubleType(), nullable=True),
])
df_cm = spark.read.csv(f"{ORIGEN_DATOS}/campo_magnetico.csv", header=True, schema=schema_cm)
print(f"Campo magnetico: {df_cm.count():,} registros")

schema_va = StructType([
    StructField("TempOut", DoubleType(), nullable=True),
    StructField("OutHum", DoubleType(), nullable=True),
    StructField("WindSpeed", DoubleType(), nullable=True),
    StructField("WindDir", DoubleType(), nullable=True),
    StructField("Bar", DoubleType(), nullable=True),
    StructField("Rain", DoubleType(), nullable=True),
    StructField("SolarRad", DoubleType(), nullable=True),
    StructField("UVIndex", DoubleType(), nullable=True),
    StructField("FechaHora", TimestampType(), nullable=False),
])
df_va = spark.read.csv(f"{ORIGEN_DATOS}/variables_ambientales.csv", header=True, schema=schema_va)
print(f"Variables ambientales: {df_va.count():,} registros")


Campo electrico: 186,664 registros
Campo magnetico: 525,600 registros
Variables ambientales: 708,958 registros


**Error frecuente**: la fuente original trae esta columna como `SolarRad.` (con un punto al final, tal como la exporta el equipo de medicion). `col("SolarRad.")` falla con `AnalysisException` -- Spark interpreta el punto como acceso a un campo anidado (`objeto.campo`), no como parte literal del nombre. La forma mas simple de evitarlo no es escapar el nombre en cada uso, sino no arrastrarlo: como `header=True` junto con un `schema` explicito hace que Spark ignore el texto del header para nombrar columnas, basta con declarar el nombre ya limpio (`SolarRad`, sin punto) en el `StructField` de arriba -- el resto del notebook nunca ve el nombre problematico.


## 3. Resolver duplicados de `FechaHora` en variables ambientales, antes de integrar

A diferencia de S3 (donde `customer_id` no tenia duplicados), aca la fuente de variables ambientales SI trae mas de una fila para el mismo minuto -- hay que resolverlo antes de unir, o el `join` multiplicaria filas sin que nadie lo note. Mismo patron de S3 (`Window`+`row_number()`): en vez de ordenar por `age`, se ordena por la fila con **menos nulos**, para conservar la version mas completa de cada minuto.


In [4]:
from pyspark.sql.functions import col, count as spark_count, when, lit

duplicadas = df_va.count() - df_va.dropDuplicates(["FechaHora"]).count()
print(f"Filas con FechaHora duplicada en variables ambientales: {duplicadas:,}")


[Stage 12:=====>                                                   (1 + 9) / 10]

Filas con FechaHora duplicada en variables ambientales: 181,918


In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

columnas_conteo_nulos = [c for c in df_va.columns if c not in ("FechaHora", "WindDir")]

df_va_con_conteo = df_va.withColumn(
    "CantidadNulos",
    sum(when(col(c).isNull(), 1).otherwise(0) for c in columnas_conteo_nulos),
)

ventana_va = Window.partitionBy("FechaHora").orderBy(col("CantidadNulos").asc())

df_va_unico = (
    df_va_con_conteo
    .withColumn("row_num", row_number().over(ventana_va))
    .filter(col("row_num") == 1)
    .drop("row_num", "CantidadNulos")
)

print(f"Variables ambientales, un registro por minuto: {df_va_unico.count():,}")


26/09/02 00:57:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
[Stage 18:=================>                                       (3 + 7) / 10]

Variables ambientales, un registro por minuto: 527,040


## 4. Integrar las tres fuentes

`FechaHora` es la clave comun. El campo electrico queda como tabla principal (`left join`): interesa el periodo que ese sensor cubre, no el de los otros dos.


In [6]:
df_integrado = (
    df_ce
    .join(df_cm, on="FechaHora", how="left")
    .join(df_va_unico, on="FechaHora", how="left")
)

print(f"Integrado: {df_integrado.count():,} registros x {len(df_integrado.columns)} columnas")
df_integrado.printSchema()


26/09/02 00:57:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
                                                                                

Integrado: 186,664 registros x 11 columnas
root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- WindDir: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)



## 5. Explorar nulos en la tabla integrada

Mismo control de calidad de S3 (2.2.3), aplicado a la tabla recien integrada -- un `left join` puede introducir nulos nuevos si algun `FechaHora` del campo electrico no tiene contraparte en las otras dos fuentes.


In [7]:
df_integrado.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in df_integrado.columns
]).show(vertical=True, truncate=False)


26/09/02 00:59:51 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
[Stage 37:>                                                         (0 + 8) / 8]

-RECORD 0-----------
 FechaHora | 0      
 Valor_CE  | 0      
 Valor_CM  | 0      
 TempOut   | 0      
 OutHum    | 0      
 WindSpeed | 0      
 WindDir   | 186664 
 Bar       | 0      
 Rain      | 0      
 SolarRad  | 0      
 UVIndex   | 0      



## 6. Tratar `WindDir` (100% nula) y el codigo de error `99999`

`WindDir` no es un caso de "algunos nulos" como en H&M -- es una columna sin un solo valor util en las 708 958 filas originales. Una columna así no aporta nada al analisis; se descarta, no se rellena.

`Valor_CM` trae otro tipo de problema, distinto de un nulo: el sensor de campo magnetico usa `99999` como codigo de error del equipo, no como un valor fisico real. `isNull()` no lo detecta -- hay que conocer el dominio del dato para encontrarlo.


In [8]:
nulos_winddir = df_integrado.filter(col("WindDir").isNull()).count()
total = df_integrado.count()
print(f"WindDir nula: {nulos_winddir:,} de {total:,} ({nulos_winddir/total*100:.1f}%)")

df_sin_winddir = df_integrado.drop("WindDir")


26/09/02 00:59:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
26/09/02 00:59:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
                                                                                

WindDir nula: 186,664 de 186,664 (100.0%)


In [9]:
errores_cm = df_sin_winddir.filter(col("Valor_CM") == 99999).count()
print(f"Filas con codigo de error Valor_CM=99999: {errores_cm:,}")

df_limpio = df_sin_winddir.filter(col("Valor_CM") != 99999)
print(f"Filas despues de eliminar el codigo de error: {df_limpio.count():,}")


26/09/02 01:00:01 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv


Filas con codigo de error Valor_CM=99999: 2,126


26/09/02 01:00:02 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv


Filas despues de eliminar el codigo de error: 184,538


## 7. Confirmar ausencia de duplicados en la tabla final

El campo electrico (tabla principal) ya no tenia `FechaHora` duplicada, y las ambientales se resolvieron en el paso 3 -- confirma que el resultado del `join` tampoco los introdujo.


In [10]:
total_final = df_limpio.count()
sin_duplicar = df_limpio.dropDuplicates(["FechaHora"]).count()

print(f"Total: {total_final:,}, sin duplicar por FechaHora: {sin_duplicar:,}")
assert total_final == sin_duplicar, "Hay FechaHora duplicada en la tabla integrada final"


26/09/02 01:00:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
                                                                                

Total: 184,538, sin duplicar por FechaHora: 184,538


## 8. Nulos finales sobre las 9 variables y filtrado

Las 9 variables (`Valor_CE`, `Valor_CM` y las 7 ambientales restantes, sin `WindDir`) son todas necesarias para el modelo de regresion de S4 -- una fila con cualquiera de ellas en nulo no sirve como entrada. A diferencia de H&M (donde rellenar `FN`/`Active` con 0 tenia sentido), aca ninguna de las 9 variables fisicas admite un relleno razonable: un `0` en `TempOut` no es "temperatura ausente", es una temperatura falsa. Se descartan las filas incompletas con `.na.drop(subset=[...])`, no se rellenan.


In [11]:
VARIABLES_9 = [
    "Valor_CE", "Valor_CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "Rain", "SolarRad", "UVIndex",
]

antes = df_limpio.count()
df_valido = df_limpio.na.drop(subset=VARIABLES_9)
despues = df_valido.count()

print(f"Filas antes: {antes:,}, despues de na.drop(subset=VARIABLES_9): {despues:,}")
print(f"Filas eliminadas por nulos en variables criticas: {antes - despues:,}")

df_valido = df_valido.cache()


26/09/02 01:00:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
26/09/02 01:00:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
                                                                                

Filas antes: 184,538, despues de na.drop(subset=VARIABLES_9): 184,538
Filas eliminadas por nulos en variables criticas: 0


## 9. Escritura particionada en Parquet, por mes

H&M particiono por `club_member_status` (una columna categorica ya presente en los datos). Aca no existe una columna categorica natural -- pero `FechaHora` sí permite **derivar** una: el mes (`AnioMes`). Particionar series de tiempo por periodo (mes, dia) es el patron mas comun en almacenamiento analitico real, distinto del patron "particionar por categoria" de S3, pero basado en la misma idea: pocos valores distintos, usados seguido en filtros.


In [12]:
from pyspark.sql.functions import date_format

df_particionable = df_valido.withColumn("AnioMes", date_format(col("FechaHora"), "yyyy-MM"))

(
    df_particionable
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("AnioMes")
    .save(f"{ARTIFACTS}/campo_electrico_particionado")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/campo_electrico_particionado")):
    print(carpeta)


26/09/02 01:00:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s03b-calidad-campo-electrico/data/variables_ambientales.csv
                                                                                

._SUCCESS.crc
AnioMes=2025-05
AnioMes=2025-06
AnioMes=2025-07
AnioMes=2025-08
AnioMes=2025-09
AnioMes=2025-10
AnioMes=2025-11
AnioMes=2025-12
_SUCCESS


## 10. Leer de vuelta y verificar el particionamiento


In [13]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/campo_electrico_particionado")
df_verificacion.printSchema()

assert df_verificacion.count() == df_particionable.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

df_verificacion.filter(col("AnioMes") == "2025-09").explain(True)


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)

Verificado: 184,538 filas, ida y vuelta sin perdida.
== Parsed Logical Plan ==
'Filter '`=`('AnioMes, 2025-09)
+- Relation [FechaHora#779,Valor_CE#780,Valor_CM#781,TempOut#782,OutHum#783,WindSpeed#784,Bar#785,Rain#786,SolarRad#787,UVIndex#788,AnioMes#789] parquet

== Analyzed Logical Plan ==
FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double, AnioMes: string
Filter (AnioMes#789 = 2025-09)
+- Relation [FechaHora#779,Valor_CE#780,Valor_CM#78

In [14]:
df_valido.unpersist()


DataFrame[FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double]

## 11. Documentar hallazgos y responder preguntas de reflexion

**Reflexion tecnica breve** (5 a 8 lineas): ?por que resolver los duplicados de variables ambientales ANTES del `join` (paso 3) evita un problema que aparecer­a despues, mas dificil de rastrear? ?que hubiera pasado si trataras `Valor_CM=99999` como si fuera un valor fisico real, sin filtrarlo? ?por que particionar por `AnioMes` tiene sentido aca, y por que no serviria particionar por `FechaHora` completa (minuto a minuto)?
